# Playground · K-Nearest Neighbors (KNN)

**Tópicos de Inteligencia de Negocios** · Clasificación supervisada

> KNN no entrena un modelo; **memoriza** los datos. Cuando llega un punto nuevo, vota entre sus K vecinos más cercanos. Acá podrás ver cómo cambia su comportamiento al variar **K**, la **métrica de distancia** y los **pesos**.

¿Qué vas a poder hacer aquí?

1. Generar datasets 2D de clasificación (binaria o multiclase).
2. Variar K, distancia (euclidiana / Manhattan / Minkowski) y pesos (uniforme / por distancia).
3. Visualizar la **frontera de decisión jagged** característica de KNN.
4. Subir tu CSV.

> 🔍 **Tip:** los botones **`?`** explican cada parámetro. Configura → presiona **🚀 Entrenar modelo**.


## Marco Teórico

### El algoritmo
1. **Memoriza** todos los puntos de entrenamiento (sin entrenar nada).
2. Cuando llega un punto nuevo $x_{nuevo}$:
   - Calcula la distancia entre $x_{nuevo}$ y todos los puntos de train.
   - Encuentra los **K más cercanos**.
   - **Vota**: la clase mayoritaria gana (o, con `weights='distance'`, los más cercanos pesan más).

### Distancias

| Métrica | Fórmula |
|--------|---------|
| **Euclidiana** | $\sqrt{\sum (x_i - y_i)^2}$ |
| **Manhattan** | $\sum \mid x_i - y_i \mid$ |
| **Minkowski (p)** | $\left( \sum \mid x_i - y_i \mid^p \right)^{1/p}$ |

> 💡 Minkowski con p=2 → Euclidiana. Con p=1 → Manhattan.

### El valor de K

| K | Comportamiento |
|---|----------------|
| **K = 1** | Muy sensible al ruido (sobreajusta) — frontera caótica |
| **K pequeño (3-5)** | Balance, captura matices locales |
| **K = √n** | Regla práctica — buen punto de partida |
| **K grande** | Suaviza la frontera; puede subajustar (ignora estructura local) |

> ⚠️ **Importante:** las distancias son sensibles a la escala. **Siempre normaliza** las features (en este playground ya está hecho con `StandardScaler`).


## 1. Configuración inicial

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification, make_moons, make_blobs
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams['figure.dpi'] = 90
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('✅ Listo. Librerías cargadas.')


## 2. Funciones auxiliares

In [ ]:
def con_ayuda(control, explicacion):
    btn = widgets.Button(description='?', button_style='info', tooltip=explicacion,
                         layout=widgets.Layout(width='30px', height='28px', margin='0 0 0 4px'))
    panel = widgets.HTML(
        value=(f'<div style="background:#dbeafe; padding:8px 10px; border-radius:4px; '
               f'border-left:3px solid #2563eb; margin:2px 0 8px 18px; font-size:12px; '
               f'color:#1e3a8a;">💡 {explicacion}</div>'),
        layout=widgets.Layout(display='none'))
    btn.on_click(lambda _: setattr(panel.layout, 'display',
                                   'none' if panel.layout.display != 'none' else 'block'))
    return widgets.VBox([widgets.HBox([control, btn]), panel])


def generar_dataset(tipo='Lunas', n=200, ruido=0.2, n_clases=2, seed=42):
    if tipo == 'Lunas':
        X, y = make_moons(n_samples=n, noise=ruido, random_state=seed)
    elif tipo == 'Blobs':
        X, y = make_blobs(n_samples=n, centers=n_clases,
                          cluster_std=ruido*4 + 0.5, random_state=seed)
    elif tipo == 'Classification':
        X, y = make_classification(n_samples=n, n_features=2, n_redundant=0,
                                   n_informative=2, n_clusters_per_class=1,
                                   n_classes=n_clases, flip_y=ruido*0.3,
                                   class_sep=2.0 - ruido, random_state=seed)
    return X, y


def construir_knn(k, metrica, weights, p=2):
    metric_kwargs = {}
    if metrica == 'minkowski':
        metric_kwargs = {'metric': 'minkowski', 'p': p}
    else:
        metric_kwargs = {'metric': metrica}
    return Pipeline([
        ('scaler', StandardScaler()),
        ('modelo', KNeighborsClassifier(n_neighbors=k, weights=weights, **metric_kwargs)),
    ])


def graficar_resultados_knn(X, y, pipeline, X_train, X_test, y_train, y_test, titulo=''):
    fig = plt.figure(figsize=(14, 7))
    gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)
    ax1 = fig.add_subplot(gs[:, :2])
    ax2 = fig.add_subplot(gs[0, 2])
    ax3 = fig.add_subplot(gs[1, 2])

    # Frontera de decisión
    margen = 0.5
    xx, yy = np.meshgrid(
        np.linspace(X[:,0].min()-margen, X[:,0].max()+margen, 200),
        np.linspace(X[:,1].min()-margen, X[:,1].max()+margen, 200))
    Z = pipeline.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    n_clases = len(np.unique(y))
    cmap_back = plt.cm.RdYlBu_r if n_clases == 2 else plt.cm.Set3
    ax1.contourf(xx, yy, Z, alpha=0.35, cmap=cmap_back)
    cmap_pts = plt.cm.RdYlBu_r if n_clases == 2 else plt.cm.Set1
    ax1.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap=cmap_pts, s=40,
                edgecolor='white', linewidth=0.5, alpha=0.85, label='Train')
    ax1.scatter(X_test[:,0], X_test[:,1], c=y_test, cmap=cmap_pts, s=80,
                marker='s', edgecolor='black', linewidth=1.0, alpha=0.95, label='Test')
    ax1.set_title(f'Frontera de decisión KNN {titulo}', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Feature 1'); ax1.set_ylabel('Feature 2')
    ax1.legend(loc='best')

    # Matriz de confusión
    y_pred = pipeline.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    ax2.imshow(cm, cmap='Blues', aspect='auto')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax2.text(j, i, cm[i,j], ha='center', va='center',
                     color='white' if cm[i,j] > cm.max()/2 else 'black',
                     fontsize=12, fontweight='bold')
    ax2.set_title('Matriz de Confusión (Test)', fontsize=11, fontweight='bold')
    ax2.set_xticks(range(cm.shape[1])); ax2.set_yticks(range(cm.shape[0]))
    ax2.set_xticklabels([f'Pred {i}' for i in range(cm.shape[1])])
    ax2.set_yticklabels([f'Real {i}' for i in range(cm.shape[0])])
    ax2.grid(False)

    # Métricas
    avg = 'binary' if n_clases == 2 else 'macro'
    metricas = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precisión': precision_score(y_test, y_pred, average=avg, zero_division=0),
        'Recall': recall_score(y_test, y_pred, average=avg, zero_division=0),
        'F1': f1_score(y_test, y_pred, average=avg, zero_division=0),
    }
    ax3.axis('off')
    txt = '\n'.join([f'{k:<12s}: {v:.4f}' for k, v in metricas.items()])
    ax3.text(0.05, 0.9, '📊 Métricas (Test)', transform=ax3.transAxes,
             fontsize=11, fontweight='bold', va='top')
    ax3.text(0.05, 0.7, txt, transform=ax3.transAxes, fontsize=11,
             va='top', family='monospace')

    plt.tight_layout(); plt.show()


## 3. Playground · Datos sintéticos 🧪

> 🔍 Botones **`?`** para ayuda. Configura → **🚀 Entrenar modelo**.


In [ ]:
# DATOS
w_tipo = widgets.Dropdown(options=['Lunas','Blobs','Classification'],
                          value='Lunas', description='Dataset:',
                          style={'description_width':'110px'})
w_n = widgets.IntSlider(value=200, min=50, max=600, step=10,
                        description='n puntos:', style={'description_width':'110px'})
w_ruido = widgets.FloatSlider(value=0.2, min=0.0, max=0.6, step=0.05,
                              description='Ruido:', style={'description_width':'110px'})
w_clases = widgets.IntSlider(value=2, min=2, max=4, step=1,
                             description='Clases:', style={'description_width':'110px'})
w_seed = widgets.IntSlider(value=42, min=0, max=100,
                           description='Semilla:', style={'description_width':'110px'})

# MODELO
w_k = widgets.IntSlider(value=5, min=1, max=50, step=1,
                        description='K vecinos:', style={'description_width':'110px'})
w_metrica = widgets.Dropdown(options=['euclidean','manhattan','minkowski','chebyshev'],
                             value='euclidean', description='Distancia:',
                             style={'description_width':'110px'})
w_p = widgets.IntSlider(value=2, min=1, max=5,
                        description='p (Minkowski):', style={'description_width':'110px'})
w_weights = widgets.Dropdown(options=['uniform','distance'], value='uniform',
                             description='Pesos:', style={'description_width':'110px'})
w_split = widgets.FloatSlider(value=0.25, min=0.1, max=0.5, step=0.05,
                              description='Test size:', style={'description_width':'110px'})

# Explicaciones
ay_tipo = ('Forma del dataset. Lunas y Classification ponen a prueba la flexibilidad de KNN; '
           'Blobs son lo más fácil.')
ay_n = 'Cantidad de puntos. KNN guarda TODOS, así que muchos puntos = predicción más lenta.'
ay_ruido = 'Cuánto se solapan las clases. Más ruido = clasificación más difícil.'
ay_clases = 'Número de clases (solo aplica a Blobs y Classification).'
ay_seed = 'Misma semilla = mismos datos.'
ay_k = ('Número de vecinos a consultar. K=1 es muy sensible al ruido. '
        'K grande suaviza pero puede subajustar. Una regla práctica: K ≈ √n (impar para evitar empates).')
ay_metrica = ('Cómo mide cercanía. Euclidean = línea recta. Manhattan = a lo "taxi" (en cuadrícula). '
              'Minkowski generaliza ambas. Chebyshev = máximo de las diferencias por eje.')
ay_p = 'Parámetro de Minkowski. p=1 → Manhattan. p=2 → Euclidiana. p>2 → métricas más extremas.'
ay_weights = ('uniform: todos los K vecinos votan igual. distance: vecinos más cercanos pesan más. '
              'Distance suele dar fronteras más suaves cerca de las clases.')
ay_split = '0.25 = 75% train / 25% test.'

panel_d = widgets.VBox([
    widgets.HTML('<b>📊 Datos</b>'),
    con_ayuda(w_tipo, ay_tipo), con_ayuda(w_n, ay_n),
    con_ayuda(w_ruido, ay_ruido), con_ayuda(w_clases, ay_clases),
    con_ayuda(w_seed, ay_seed),
])
panel_m = widgets.VBox([
    widgets.HTML('<b>🧮 Modelo</b>'),
    con_ayuda(w_k, ay_k), con_ayuda(w_metrica, ay_metrica),
    con_ayuda(w_p, ay_p), con_ayuda(w_weights, ay_weights),
    con_ayuda(w_split, ay_split),
])
controles = widgets.HBox([panel_d, panel_m])
salida = widgets.Output()

def actualizar():
    with salida:
        clear_output(wait=True)
        try:
            X, y = generar_dataset(w_tipo.value, w_n.value, w_ruido.value,
                                    w_clases.value, w_seed.value)
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=w_split.value,
                                                       random_state=42, stratify=y)
            pipe = construir_knn(w_k.value, w_metrica.value, w_weights.value, w_p.value)
            pipe.fit(X_tr, y_tr)
            graficar_resultados_knn(X, y, pipe, X_tr, X_te, y_tr, y_te,
                                     titulo=f'· K={w_k.value}, {w_metrica.value}, {w_weights.value}')
        except Exception as e:
            print(f'⚠️ {e}')

btn_e = widgets.Button(description='🚀 Entrenar modelo', button_style='primary',
                       layout=widgets.Layout(width='220px', height='40px',
                                             margin='10px 0 6px 0'))
estado = widgets.HTML(value='<span style="color:#64748b;font-style:italic;">'
                            'Configura los parámetros y haz clic en <b>Entrenar modelo</b>.</span>')

def _stale(*_):
    estado.value = ('<span style="color:#ea580c;">🔄 <b>Cambios sin aplicar.</b> '
                    'Haz clic en <b>Entrenar modelo</b>.</span>')

def _go(_):
    btn_e.disabled = True; btn_e.description = '⏳ Entrenando...'
    estado.value = '<span style="color:#2563eb;">⏳ Entrenando...</span>'
    try:
        actualizar()
        estado.value = ('<span style="color:#16a34a;">✅ <b>Modelo entrenado.</b> '
                        'Cambia parámetros y vuelve a entrenar.</span>')
    except Exception as e:
        estado.value = f'<span style="color:#dc2626;">❌ {e}</span>'
    finally:
        btn_e.disabled = False; btn_e.description = '🚀 Entrenar modelo'

btn_e.on_click(_go)
for w in [w_tipo, w_n, w_ruido, w_clases, w_seed, w_k, w_metrica, w_p, w_weights, w_split]:
    w.observe(_stale, names='value')

display(controles, btn_e, estado, salida)
actualizar()
estado.value = ('<span style="color:#16a34a;">✅ <b>Modelo entrenado con la configuración inicial.</b></span>')


## 4. Playground · Sube tu CSV 📂

Sube un CSV con 2 columnas numéricas (features) y 1 con etiqueta (puede ser binaria o multiclase).


In [ ]:
estado_csv = {'df': None}

w_upload = widgets.FileUpload(accept='.csv', multiple=False, description='📁 Subir CSV')
w_x1 = widgets.Dropdown(options=[], description='Feature 1:', style={'description_width':'110px'})
w_x2 = widgets.Dropdown(options=[], description='Feature 2:', style={'description_width':'110px'})
w_yc = widgets.Dropdown(options=[], description='Etiqueta:', style={'description_width':'110px'})

w_k2 = widgets.IntSlider(value=5, min=1, max=50,
                         description='K vecinos:', style={'description_width':'110px'})
w_m2 = widgets.Dropdown(options=['euclidean','manhattan','minkowski','chebyshev'],
                        value='euclidean', description='Distancia:',
                        style={'description_width':'110px'})
w_w2 = widgets.Dropdown(options=['uniform','distance'], value='uniform',
                        description='Pesos:', style={'description_width':'110px'})
w_s2 = widgets.FloatSlider(value=0.25, min=0.1, max=0.5, step=0.05,
                           description='Test size:', style={'description_width':'110px'})

salida_csv = widgets.Output()
salida_info = widgets.Output()

def on_upload(change):
    with salida_info:
        clear_output(wait=True)
        if not w_upload.value: return
        try:
            archivo = w_upload.value[0] if isinstance(w_upload.value, tuple) else next(iter(w_upload.value.values()))
            contenido = archivo['content']
            nombre = archivo.get('name','archivo.csv')
        except Exception:
            archivo = list(w_upload.value.values())[0]
            contenido = archivo['content']
            nombre = archivo.get('metadata',{}).get('name','archivo.csv')
        try:
            df = pd.read_csv(io.BytesIO(bytes(contenido)))
        except Exception as e:
            print(f'⚠️ {e}'); return
        estado_csv['df'] = df
        cn = df.select_dtypes(include=[np.number]).columns.tolist()
        if len(cn) < 3:
            print(f'⚠️ Necesitas ≥ 3 columnas numéricas. Encontré: {cn}'); return
        w_x1.options = cn; w_x2.options = cn; w_yc.options = cn
        w_x1.value = cn[0]; w_x2.value = cn[1]; w_yc.value = cn[-1]
        print(f'✅ {nombre} — {df.shape[0]}×{df.shape[1]}')
        display(df.head())

w_upload.observe(on_upload, names='value')

def actualizar2():
    with salida_csv:
        clear_output(wait=True)
        df = estado_csv['df']
        if df is None: print('⬆️ Sube un CSV primero.'); return
        try:
            sub = df[[w_x1.value, w_x2.value, w_yc.value]].dropna()
            X = sub[[w_x1.value, w_x2.value]].values
            y_raw = sub[w_yc.value].values
            unicos, y = np.unique(y_raw, return_inverse=True)
            if len(unicos) < 2:
                print('⚠️ La etiqueta debe tener al menos 2 valores.'); return
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=w_s2.value,
                                                       random_state=42, stratify=y)
            pipe = construir_knn(w_k2.value, w_m2.value, w_w2.value)
            pipe.fit(X_tr, y_tr)
            graficar_resultados_knn(X, y, pipe, X_tr, X_te, y_tr, y_te,
                                     titulo=f'· K={w_k2.value}')
        except Exception as e:
            print(f'⚠️ {e}')

btn_e2 = widgets.Button(description='🚀 Entrenar modelo', button_style='primary',
                        layout=widgets.Layout(width='220px', height='40px',
                                              margin='10px 0 6px 0'))
estado2 = widgets.HTML(value='<span style="color:#64748b;font-style:italic;">'
                             'Sube un CSV y haz clic en <b>Entrenar modelo</b>.</span>')

def _stale2(*_):
    if estado_csv['df'] is None: return
    estado2.value = ('<span style="color:#ea580c;">🔄 <b>Cambios sin aplicar.</b></span>')

def _go2(_):
    if estado_csv['df'] is None:
        estado2.value = '<span style="color:#dc2626;">⚠️ Primero sube un CSV.</span>'; return
    btn_e2.disabled = True; btn_e2.description = '⏳ Entrenando...'
    estado2.value = '<span style="color:#2563eb;">⏳ Entrenando...</span>'
    try:
        actualizar2()
        estado2.value = ('<span style="color:#16a34a;">✅ <b>Modelo entrenado.</b></span>')
    except Exception as e:
        estado2.value = f'<span style="color:#dc2626;">❌ {e}</span>'
    finally:
        btn_e2.disabled = False; btn_e2.description = '🚀 Entrenar modelo'

btn_e2.on_click(_go2)
for w in [w_x1, w_x2, w_yc, w_k2, w_m2, w_w2, w_s2]:
    w.observe(_stale2, names='value')

panel_c = widgets.VBox([
    widgets.HTML('<b>📁 Datos (CSV)</b>'),
    con_ayuda(w_upload, 'Sube tu .csv (2 features numéricas + 1 etiqueta).'),
    con_ayuda(w_x1, 'Primera variable predictora.'),
    con_ayuda(w_x2, 'Segunda variable predictora.'),
    con_ayuda(w_yc, 'Variable a predecir.'),
])
panel_m2 = widgets.VBox([
    widgets.HTML('<b>🧮 Modelo</b>'),
    con_ayuda(w_k2, ay_k), con_ayuda(w_m2, ay_metrica),
    con_ayuda(w_w2, ay_weights), con_ayuda(w_s2, ay_split),
])
display(widgets.HBox([panel_c, panel_m2]), btn_e2, estado2, salida_info, salida_csv)


## 5. Ejercicios guiados 📝

### Ejercicio 1 — La curva del K
1. Dataset = **Lunas**, n = 200, ruido = 0.25.
2. Prueba K = 1, 3, 7, 15, 35, 50.
3. **Pregunta:** ¿Cómo cambia la frontera? ¿En qué K notas más sobreajuste y en cuál más subajuste?

### Ejercicio 2 — La regla de √n
1. Calcula √(n_train) (con n=200 y test=0.25, train=150 → √150 ≈ 12). Usa el K impar más cercano (K=11 o K=13).
2. Compáralo con K=1 y K=50.
3. **Pregunta:** ¿La regla práctica te da un buen K?

### Ejercicio 3 — Distancia importa
1. Mismos datos. Mantén K=5.
2. Prueba métrica euclidean, manhattan, chebyshev.
3. **Pregunta:** ¿Cambia mucho la frontera? ¿Por qué?

### Ejercicio 4 — Pesos uniformes vs por distancia
1. Dataset = **Classification**, ruido = 0.4, K = 15.
2. Compara `weights='uniform'` vs `weights='distance'`.
3. **Pregunta:** ¿Cuál de los dos es más robusto cuando hay puntos de la otra clase cerca de la frontera?

### Ejercicio 5 — KNN multiclase
1. Dataset = **Blobs**, n_clases = 4, ruido = 0.2.
2. Prueba K = 5, 10, 20.
3. **Pregunta:** ¿KNN tiene problema con multiclase? ¿Cómo es la frontera comparada con la binaria?


## 6. Resumen

- KNN es **lazy learning**: no entrena, memoriza. Predicción O(n) en tiempo.
- **K** es el parámetro estrella: chico → alta varianza; grande → alto sesgo.
- Las **distancias son sensibles a la escala**: siempre normaliza features.
- Funciona bien con datasets pequeños/medianos y fronteras complicadas.
- En alta dimensionalidad, las distancias pierden significado (curse of dimensionality) — KNN sufre.

---

> *Tópicos de Inteligencia de Negocios · Playground de Machine Learning*
